# StudyMate · Part 8 of 8: StudyMate App

**Assignment part:** Interface  
**Group 7:** Sadia Saeed · Eva Abouabouharb · Neida Guzman

The **AI StudyMate** Gradio web app (violet theme): the student asks a question and sees the lesson used, the answer card, a hidden flashcard answer, related lessons, and the lesson's whiteboard.

> 📖 **This notebook is for reading.** It is one section of the full notebook, with the real outputs kept. Each part uses results from the parts before it, so to **run** StudyMate, open [`FN_StudyMate_Group7_ITAI2377.ipynb`](FN_StudyMate_Group7_ITAI2377.ipynb) and use **Runtime → Run all**.

## 14. StudyMate App (Gradio Interface)

**Purpose:** A simple web app where students ask an ML question and get a full answer card.

**What the student sees:**
- 📘 The lesson StudyMate used
- 💡 Simple explanation, 🏥 real-life example, 🧠 memory trick
- 🃏 A flashcard question, with the answer hidden until clicked
- 🔗 Related lessons
- ✏️ A hand-drawn whiteboard for every lesson (all 66)
- A "Sorry" message for off-topic questions (from the guard in Cell 5)

**How it works:** The app calls `answer()` from Cell 5 (two-stage search → guard → Qwen), splits the answer card into separate boxes, and plays the lesson's whiteboard from `WHITEBOARDS` (Cells 12–13H) if one exists.

**Note:** Needs a T4 GPU (for Qwen). Run Cells 11–13H before this cell. `share=True` creates a temporary public link.

In [56]:

# CODE CELL 14 — STUDYMATE APP (VIOLET INTERFACE)

!pip install -q gradio

import gradio as gr
import inspect
import re

WB_NOTE = "✏️ *No whiteboard for this lesson yet.*"


# ==============================
# APPLICATION FUNCTIONS
# ==============================

def split_card(card):
    """Split Qwen's card into its 4 sections."""
    pieces = re.split(
        r"\*\*(Simple explanation|Real-life example|Memory trick|Flashcard):\*\*",
        card
    )

    parts = {
        name: text.strip()
        for name, text in zip(pieces[1::2], pieces[2::2])
    }

    fc = re.split(
        r"\s*A:\s*",
        parts.get("Flashcard", ""),
        maxsplit=1
    )

    parts["Q"] = fc[0].replace("Q:", "").strip()
    parts["A"] = fc[1].strip() if len(fc) > 1 else ""

    return parts


def ask(question):

    if not question.strip():
        return "Please type a question first.", "", "", "", "", "", "", ""

    r = answer(question)

    if r["lesson"] == "—":
        return f"### 🙅 {SORRY}", "", "", "", "", "", "", ""

    p = split_card(r["card"])

    skipped = "(StudyMate skipped this part)"

    board = (
        WHITEBOARDS[r["lesson"]].iframe()
        if r["lesson"] in WHITEBOARDS
        else WB_NOTE
    )

    return (
        f"### 📘 Lesson used: {r['lesson']}",
        p.get("Simple explanation") or skipped,
        p.get("Real-life example") or skipped,
        p.get("Memory trick") or skipped,
        p.get("Q") or skipped,
        p.get("A") or skipped,
        f"**🔗 Related lessons:** {', '.join(r['related'])}",
        board
    )


# ==============================
# VIOLET THEME
# ==============================

theme = gr.themes.Soft(
    primary_hue="violet",
    secondary_hue="purple",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Poppins"), "Arial"]
).set(

    body_background_fill="#F5F3FF",
    body_background_fill_dark="#F5F3FF",

    body_text_color="#312E81",
    body_text_color_dark="#312E81",

    block_background_fill="#FFFFFF",
    block_background_fill_dark="#FFFFFF",

    block_label_text_color="#4C1D95",
    block_label_text_color_dark="#4C1D95",

    block_title_text_color="#4C1D95",
    block_title_text_color_dark="#4C1D95",

    input_background_fill="#FFFFFF",
    input_background_fill_dark="#FFFFFF",

    block_border_width="1px",

    block_shadow="0 5px 20px rgba(124,58,237,0.10)",

    button_primary_background_fill="#7C3AED",
    button_primary_background_fill_dark="#7C3AED",

    button_primary_text_color="#FFFFFF",
    button_primary_text_color_dark="#FFFFFF",

    button_secondary_background_fill="#EDE9FE",
    button_secondary_background_fill_dark="#EDE9FE",

    button_secondary_text_color="#4C1D95",
    button_secondary_text_color_dark="#4C1D95"
)


# ==============================
# CUSTOM CSS — CORRECTED COLORS
# ==============================

custom_css = """

.gradio-container {
    background: linear-gradient(
        135deg, #F5F3FF, #EDE9FE, #FAF5FF
    ) !important;

    max-width: 1250px !important;
    margin: auto;
    padding: 25px !important;

    color: #312E81 !important;

    --body-text-color: #312E81 !important;
    --body-text-color-subdued: #5B21B6 !important;
    --block-background-fill: #FFFFFF !important;
    --input-background-fill: #FFFFFF !important;
    --block-label-text-color: #4C1D95 !important;
    --block-title-text-color: #4C1D95 !important;
}


/* FIX HEADINGS AND TEXT VISIBILITY */

.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container p,
.gradio-container label {
    color: #4C1D95 !important;
}


/* BEAUTIFUL HEADER */

#studymate-header {
    background: linear-gradient(
        135deg, #4C1D95, #7C3AED, #A78BFA
    );

    padding: 35px;
    border-radius: 22px;
    text-align: center;
    margin-bottom: 20px;

    box-shadow: 0 8px 25px rgba(76,29,149,0.25);
}

#studymate-header h1 {
    color: #FFFFFF !important;
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 10px;
}

#studymate-header p {
    color: #FFFFFF !important;
    font-size: 16px;
}


/* ASK BUTTON */

#ask-button {
    background: linear-gradient(
        135deg, #6D28D9, #9333EA
    ) !important;

    color: #FFFFFF !important;
    border-radius: 14px !important;
    font-weight: 700 !important;
    min-height: 55px;
}

#ask-button span {
    color: #FFFFFF !important;
}

#ask-button:hover {
    box-shadow: 0 5px 18px rgba(124,58,237,0.4);
    transform: translateY(-2px);
}


/* QUESTION BOX */

#question-box {
    background: #FFFFFF !important;
    border: 2px solid #C4B5FD !important;
    border-radius: 16px !important;
}


/* FIX ALL TEXTBOXES */

.gradio-container textarea,
.gradio-container input:not([type="checkbox"]):not([type="radio"]):not([type="range"]) {
    background: #FFFFFF !important;
    color: #312E81 !important;
    -webkit-text-fill-color: #312E81 !important;
    border-radius: 12px !important;
}

.gradio-container textarea::placeholder,
.gradio-container input::placeholder {
    color: #64748B !important;
    -webkit-text-fill-color: #64748B !important;
}


/* LEARNING CARDS */

#simple-card {
    background: #EDE9FE !important;
    border-left: 5px solid #7C3AED !important;
}

#example-card {
    background: #F3E8FF !important;
    border-left: 5px solid #A855F7 !important;
}

#memory-card {
    background: #E0E7FF !important;
    border-left: 5px solid #6366F1 !important;
}

#flashcard {
    background: #FAE8FF !important;
    border-left: 5px solid #C026D3 !important;
}

#simple-card,
#example-card,
#memory-card,
#flashcard {
    border-radius: 16px !important;
    padding: 15px !important;
    box-shadow: 0 4px 12px rgba(76,29,149,0.08);
}

#simple-card label,
#example-card label,
#memory-card label,
#flashcard label {
    color: #4C1D95 !important;
}

#simple-card label span,
#example-card label span,
#memory-card label span,
#flashcard label span {
    background: #EDE9FE !important;
    color: #4C1D95 !important;
}


/* FLASHCARD ANSWER */

#flashcard-answer {
    background: #EDE9FE !important;
    color: #4C1D95 !important;
    border: 1px solid #C4B5FD !important;
    border-radius: 14px !important;
}

#flashcard-answer button,
#flashcard-answer span {
    color: #4C1D95 !important;
}


/* WHITEBOARD */

#whiteboard-panel {
    background: #FFFFFF !important;
    border: 2px solid #C4B5FD !important;
    border-radius: 20px !important;
    padding: 20px !important;
    box-shadow: 0 5px 20px rgba(124,58,237,0.12);
}

#whiteboard-panel .html-container {
    background: #FFFFFF !important;
}


/* LESSON INFORMATION */

#lesson-info {
    background: #EDE9FE !important;
    color: #4C1D95 !important;
    border-radius: 12px;
    padding: 10px;
}


/* FIX EXAMPLE QUESTIONS */

.gradio-container .dataset,
.gradio-container .dataset button,
.gradio-container .dataset span,
.gradio-container .dataset td {
    color: #4C1D95 !important;
}

.gradio-container .dataset button {
    background: #EDE9FE !important;
    border: 1px solid #C4B5FD !important;
    border-radius: 12px !important;
}

.gradio-container .dataset button:hover {
    background: #DDD6FE !important;
    color: #4C1D95 !important;
}


/* GENERAL BUTTON VISIBILITY */

.gradio-container button:not(#ask-button) {
    color: #4C1D95;
}


/* FOOTER */

#app-footer {
    text-align: center;
    color: #4C1D95 !important;
    padding: 20px;
    font-weight: 600;
}

"""


# ==============================
# GRADIO VERSION COMPATIBILITY
# ==============================

style_options = {
    "theme": theme,
    "css": custom_css
}

launch_parameters = inspect.signature(
    gr.Blocks.launch
).parameters

blocks_options = {
    k: v for k, v in style_options.items()
    if k not in launch_parameters
}

launch_options = {
    k: v for k, v in style_options.items()
    if k in launch_parameters
}


# ==============================
# STUDYMATE INTERFACE
# ==============================

with gr.Blocks(
    title="AI StudyMate",
    **blocks_options
) as app:

    gr.HTML("""
    <div id="studymate-header">
        <h1>🎓 AI StudyMate</h1>
        <p>Your Intelligent Machine Learning Companion</p>
        <p>✨ Learn • Visualize • Understand • Remember</p>
    </div>
    """)

    gr.Markdown("## 💬 What would you like to learn?")

    with gr.Row():

        question = gr.Textbox(
            label="Your question",
            scale=4,
            placeholder="Ask me any machine learning question...",
            elem_id="question-box"
        )

        ask_btn = gr.Button(
            "✨ Ask StudyMate",
            variant="primary",
            scale=1,
            elem_id="ask-button"
        )

    lesson_md = gr.Markdown(
        elem_id="lesson-info"
    )

    gr.Markdown("## 📚 Your Personalized Learning Guide")

    with gr.Row():

        simple = gr.Textbox(
            label="💡 Simple explanation",
            lines=4,
            interactive=False,
            elem_id="simple-card"
        )

        example = gr.Textbox(
            label="🌍 Real-life example",
            lines=4,
            interactive=False,
            elem_id="example-card"
        )

    with gr.Row():

        trick = gr.Textbox(
            label="🧠 Memory trick",
            lines=2,
            interactive=False,
            elem_id="memory-card"
        )

        flash_q = gr.Textbox(
            label="🃏 Flashcard question",
            lines=2,
            interactive=False,
            elem_id="flashcard"
        )

    with gr.Accordion(
        "🔮 Reveal Flashcard Answer",
        open=False,
        elem_id="flashcard-answer"
    ):

        flash_a = gr.Textbox(
            label="Answer",
            lines=2,
            interactive=False
        )

    related_md = gr.Markdown()

    gr.Markdown("## 🎨 Interactive Learning Whiteboard")

    with gr.Group(elem_id="whiteboard-panel"):

        whiteboard = gr.HTML()

    gr.Markdown("## 💭 Try These Questions")

    gr.Examples(
        [
            "How does gradient descent help a model learn?",
            "Why does my model do well on training data but badly on new data?",
            "Why do we keep some data aside for testing?",
            "How can I group customers when I don't know the categories?",
            "What is the capital of France?"
        ],
        inputs=question
    )

    gr.HTML("""
    <div id="app-footer">
        🎓 AI StudyMate | Making Machine Learning Simple
        <br>
        Learn Smarter. Visualize Better.
    </div>
    """)

    outputs = [
        lesson_md,
        simple,
        example,
        trick,
        flash_q,
        flash_a,
        related_md,
        whiteboard
    ]

    ask_btn.click(
        ask,
        inputs=question,
        outputs=outputs
    )

    question.submit(
        ask,
        inputs=question,
        outputs=outputs
    )


# ==============================
# LAUNCH APPLICATION
# ==============================

app.launch(share=True, **launch_options)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7ff2bc1670d2b1177.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
